# 03. GraphRAG on Kuzu

Этот ноутбук показывает, что vector / hybrid RAG — не единственный способ retrieval.

Вместо поиска по чанкам мы построим **property graph** из сущностей и связей, а затем ответим на вопросы через **graph retrieval + generation**.

В качестве graph-native storage здесь выбран **Kuzu**:
- он достаточно практичный, чтобы его можно было использовать дальше;
- он embedded и не требует тяжелого внешнего серверного контура;
- он использует Cypher, поэтому graph queries остаются прозрачными.


## Что будет в ноутбуке

1. Загрузим весь текущий корпус документов через уже сохраненный BM25-индекс.
2. Извлечем из чанков сущности и связи через LLM.
3. Построим property graph в `Kuzu`.
4. Покажем несколько простых Cypher-запросов к графу.
5. Соберем небольшой GraphRAG pipeline: `question -> graph retrieval -> answer`.
6. Сравним GraphRAG с flat RAG на relation-heavy вопросах.
7. В конце коротко посмотрим на другие graph / GraphRAG стеки.


## Почему здесь `Kuzu`

Для этого ноутбука нужен стек, который:
- graph-native и usable-after-course;
- не слишком тяжелый инфраструктурно;
- не превращает ноутбук в black box.

Поэтому в качестве core мы берем `Kuzu`.

Другие варианты, которые стоит знать:
- `Neo4j` — production-style graph DB: https://neo4j.com/docs/
- `LightRAG` — готовый GraphRAG framework: https://github.com/HKUDS/LightRAG
- `Microsoft GraphRAG` — более тяжелый advanced framework/pipeline: https://github.com/microsoft/graphrag
- `FalkorDBLite` — lightweight graph DB option: https://github.com/FalkorDB/falkordblite
- `RDFLib` — RDF / triples / SPARQL branch: https://rdflib.readthedocs.io/

Мы не утверждаем, что `Kuzu` — единственный правильный выбор. Это просто хороший practical middle-ground для учебного ноутбука.


In [1]:
from __future__ import annotations

import json
import os
import pickle
import re
import shutil
from pathlib import Path
from typing import Any

import chromadb
import kuzu
import pandas as pd
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_community.retrievers import BM25Retriever
from langchain_gigachat import GigaChat, GigaChatEmbeddings
from tqdm.auto import tqdm


In [2]:
repo_root_candidates = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = next(
    (path for path in repo_root_candidates if (path / '.env.example').exists()),
    Path.cwd(),
)
TOPIC7_ROOT = next(
    (
        path for path in [
            Path.cwd(),
            Path.cwd() / 'topic7_rag',
            REPO_ROOT / 'topic7_rag',
        ]
        if path.exists() and path.name == 'topic7_rag'
    ),
    REPO_ROOT / 'topic7_rag',
)
DATA_ROOT = TOPIC7_ROOT / 'data'
ENV_PATH = REPO_ROOT / '.env'

if ENV_PATH.exists():
    load_dotenv(ENV_PATH)
    env_status = str(ENV_PATH)
else:
    env_status = f"не найден ({REPO_ROOT / '.env.example'} найден)"

GIGACHAT_CREDENTIALS = os.getenv('GIGACHAT_CREDENTIALS')
GIGACHAT_SCOPE = os.getenv('GIGACHAT_SCOPE', 'GIGACHAT_API_PERS')
GIGACHAT_MODEL = os.getenv('GIGACHAT_MODEL', 'GigaChat-2-Max')
GIGACHAT_EMBEDDINGS = os.getenv('GIGACHAT_EMBEDDINGS_MODEL', 'EmbeddingsGigaR')
GIGACHAT_TIMEOUT = float(os.getenv('GIGACHAT_TIMEOUT', '60'))
GRAPH_EXTRACTION_MAX_CHUNKS = int(os.getenv('GRAPH_EXTRACTION_MAX_CHUNKS', '60'))

raw_chroma_dir = Path(os.getenv('CHROMA_PERSIST_DIR', './data/chroma_db'))
CHROMA_DIR = raw_chroma_dir if raw_chroma_dir.is_absolute() else TOPIC7_ROOT / raw_chroma_dir.relative_to(Path('./'))
BM25_PATH = DATA_ROOT / 'bm25_index.pkl'
KUZU_DB_PATH = DATA_ROOT / 'kuzu_graphrag'
EXTRACTIONS_CACHE_PATH = DATA_ROOT / 'graphrag_extractions.json'
NOTEBOOK_STATUS_PATH = DATA_ROOT / 'graphrag_run_status.json'
COLLECTION_NAME = 'course_rag'

PHOENIX_PROJECT_NAME = os.getenv('PHOENIX_PROJECT_NAME_03', 'topic7-rag-03')
PHOENIX_COLLECTOR_ENDPOINT = os.getenv('PHOENIX_COLLECTOR_ENDPOINT')
PHOENIX_WORKING_DIR = os.getenv('PHOENIX_WORKING_DIR', str(REPO_ROOT / '.phoenix'))
PHOENIX_PORT = int(os.getenv('PHOENIX_PORT', '6006'))
PHOENIX_GRPC_PORT = int(os.getenv('PHOENIX_GRPC_PORT', '4317'))
PHOENIX_ENABLED = False
PHOENIX_SESSION = None
PHOENIX_SESSION_URL = None

print('✓ Окружение загружено')
print(f'  Env file:         {env_status}')
print(f'  Topic7 root:      {TOPIC7_ROOT}')
print(f'  Chroma dir:       {CHROMA_DIR}')
print(f'  BM25 path:        {BM25_PATH}')
print(f'  Kuzu db path:     {KUZU_DB_PATH}')
print(f'  Extractions path: {EXTRACTIONS_CACHE_PATH}')
print(f'  Run status path:   {NOTEBOOK_STATUS_PATH}')
print(f'  Phoenix project:  {PHOENIX_PROJECT_NAME}')
print(f'  Phoenix working:  {PHOENIX_WORKING_DIR}')
print(f'  GigaChat timeout:  {GIGACHAT_TIMEOUT}s')
print(f'  Graph chunks cap:  {GRAPH_EXTRACTION_MAX_CHUNKS if GRAPH_EXTRACTION_MAX_CHUNKS > 0 else "all"}')


✓ Окружение загружено
  Env file:         /Users/Sergej/claude_code/agent_course/rag_course_repo/.env
  Topic7 root:      /Users/Sergej/claude_code/agent_course/rag_course_repo/topic7_rag
  Chroma dir:       /Users/Sergej/claude_code/agent_course/rag_course_repo/topic7_rag/data/chroma_db
  BM25 path:        /Users/Sergej/claude_code/agent_course/rag_course_repo/topic7_rag/data/bm25_index.pkl
  Kuzu db path:     /Users/Sergej/claude_code/agent_course/rag_course_repo/topic7_rag/data/kuzu_graphrag
  Extractions path: /Users/Sergej/claude_code/agent_course/rag_course_repo/topic7_rag/data/graphrag_extractions.json
  Run status path:   /Users/Sergej/claude_code/agent_course/rag_course_repo/topic7_rag/data/graphrag_run_status.json
  Phoenix project:  topic7-rag-03
  Phoenix working:  ./.phoenix
  GigaChat timeout:  60.0s
  Graph chunks cap:  60


In [3]:
from datetime import datetime, timezone


def write_notebook_status(stage: str, detail: str = '', status: str = 'running', **extra: Any) -> None:
    """Пишет текущий stage в JSON-файл и печатает короткий статус.

    Это помогает понять, где находится выполнение, если Jupyter/VS Code страницу обновили
    и frontend больше не показывает текущую активную ячейку.
    """
    payload = {
        'updated_at': datetime.now(timezone.utc).isoformat(),
        'stage': stage,
        'status': status,
        'detail': detail,
        **extra,
    }
    NOTEBOOK_STATUS_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(NOTEBOOK_STATUS_PATH, 'w', encoding='utf-8') as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    prefix = '▶' if status == 'running' else '✓' if status == 'done' else '⚠'
    message = f'{prefix} [{stage}] {status}'
    if detail:
        message += f': {detail}'
    print(message)


## Phoenix и tracing

В этом ноутбуке Phoenix нужен прежде всего для трассировки:
- extraction LLM-вызовов при построении графа;
- question entity extraction;
- финальной graph-based answer generation.

Режимы работы те же, что и в `02`:
- внешний collector через `PHOENIX_COLLECTOR_ENDPOINT`;
- локальный Phoenix через `phoenix.launch_app(use_temp_dir=False)`.

Важно:
- локальная Phoenix DB хранится в `PHOENIX_WORKING_DIR`;
- у этого ноутбука отдельный project name: `PHOENIX_PROJECT_NAME_03`.


In [4]:
try:
    import socket
    from urllib.error import HTTPError, URLError
    from urllib.request import urlopen

    import phoenix as px
    from phoenix.otel import register
    from openinference.instrumentation.langchain import LangChainInstrumentor

    def is_port_open(host: str, port: int, timeout: float = 0.5) -> bool:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            sock.settimeout(timeout)
            return sock.connect_ex((host, port)) == 0

    def check_phoenix_ui(url: str) -> None:
        try:
            with urlopen(url, timeout=2) as response:
                print(f"Phoenix UI отвечает: HTTP {response.status}")
        except HTTPError as exc:
            print(f"Phoenix UI вернул HTTP {exc.code}; tracing endpoint всё равно может работать.")
        except URLError as exc:
            print(f"Phoenix UI пока не ответил ({type(exc.reason).__name__}); попробуй обновить страницу через несколько секунд.")
        except Exception as exc:
            print(f"Phoenix UI check пропущен: {type(exc).__name__}: {exc}")

    if PHOENIX_COLLECTOR_ENDPOINT:
        collector_endpoint = PHOENIX_COLLECTOR_ENDPOINT.rstrip('/')
        if not collector_endpoint.endswith('/v1/traces'):
            collector_endpoint = collector_endpoint + '/v1/traces'
        PHOENIX_SESSION_URL = None
        print(f"Используем внешний Phoenix collector: {collector_endpoint}")
    else:
        local_ui_url = f"http://localhost:{PHOENIX_PORT}/"
        collector_endpoint = f"http://127.0.0.1:{PHOENIX_PORT}/v1/traces"

        if is_port_open('127.0.0.1', PHOENIX_PORT):
            # Phoenix уже запущен в этом или другом notebook/kernel.
            # Не стартуем второй instance, а подключаемся к существующему HTTP collector.
            PHOENIX_SESSION_URL = local_ui_url
            print(f"Используем уже запущенный локальный Phoenix: {PHOENIX_SESSION_URL}")
            check_phoenix_ui(PHOENIX_SESSION_URL)
        else:
            if is_port_open('127.0.0.1', PHOENIX_GRPC_PORT):
                raise RuntimeError(
                    f"gRPC port {PHOENIX_GRPC_PORT} уже занят, а HTTP port {PHOENIX_PORT} свободен. "
                    "Скорее всего остался полузапущенный Phoenix process. "
                    "Перезапусти kernel или поменяй PHOENIX_PORT/PHOENIX_GRPC_PORT в .env."
                )
            os.environ['PHOENIX_PORT'] = str(PHOENIX_PORT)
            os.environ['PHOENIX_GRPC_PORT'] = str(PHOENIX_GRPC_PORT)
            os.environ['PHOENIX_WORKING_DIR'] = PHOENIX_WORKING_DIR
            Path(PHOENIX_WORKING_DIR).mkdir(parents=True, exist_ok=True)
            PHOENIX_SESSION = px.launch_app(use_temp_dir=False)
            if PHOENIX_SESSION is None:
                raise RuntimeError('Phoenix launch_app() не смог поднять локальную сессию')
            PHOENIX_SESSION_URL = PHOENIX_SESSION.url
            collector_endpoint = f"http://127.0.0.1:{PHOENIX_SESSION.port}/v1/traces"
            print(f"Локальный Phoenix запущен: {PHOENIX_SESSION_URL}")
            print(f"Phoenix DB dir: {PHOENIX_WORKING_DIR}")

    tracer_provider = register(
        endpoint=collector_endpoint,
        protocol='http/protobuf',
        project_name=PHOENIX_PROJECT_NAME,
        batch=False,
        auto_instrument=False,
    )
    LangChainInstrumentor().instrument(tracer_provider=tracer_provider)
    PHOENIX_ENABLED = True
    print(f"✓ Phoenix tracing включен: {collector_endpoint}")
except Exception as exc:
    print('Phoenix tracing пропущен:', type(exc).__name__, exc)
    print('Если нужен tracing, проверь kernel, занятые порты и endpoint collector-а.')


/Users/Sergej/claude_code/agent_course/rag_course_repo/.venv/lib/python3.12/site-packages/authlib/_joserfc_helpers.py:8: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose import ECKey


🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
💽 Your data is being persisted to sqlite:////Users/Sergej/claude_code/agent_course/rag_course_repo/topic7_rag/.phoenix/phoenix.db
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix
Локальный Phoenix запущен: http://localhost:6006/
Phoenix DB dir: ./.phoenix
🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: topic7-rag-03
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://127.0.0.1:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

✓ Phoenix tracing включен: http://127.0.0.1:6006/v1/traces


## 0. Загрузка моделей и всего корпуса

В отличие от предыдущих идей с маленьким подкорпусом, здесь мы используем **все текущие документы**.

Практически это удобно сделать через уже сохраненный `BM25Retriever`: в нем лежат те же чанки с метаданными, которые были получены в `01_ingestion_pipeline.ipynb`.


In [5]:
embeddings = GigaChatEmbeddings(
    credentials=GIGACHAT_CREDENTIALS,
    scope=GIGACHAT_SCOPE,
    model=GIGACHAT_EMBEDDINGS,
    timeout=GIGACHAT_TIMEOUT,
    verify_ssl_certs=False,
)

llm = GigaChat(
    credentials=GIGACHAT_CREDENTIALS,
    scope=GIGACHAT_SCOPE,
    model=GIGACHAT_MODEL,
    temperature=0,
    timeout=GIGACHAT_TIMEOUT,
    verify_ssl_certs=False,
)

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
vectorstore = Chroma(
    client=chroma_client,
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
)

with open(BM25_PATH, 'rb') as f:
    bm25_retriever: BM25Retriever = pickle.load(f)

all_docs: list[Document] = list(bm25_retriever.docs)
unique_files = sorted({doc.metadata.get('filename', 'unknown') for doc in all_docs})

print('✓ Модели и документы загружены')
print(f'  Chunks: {len(all_docs)}')
print(f'  Files:  {len(unique_files)}')
for filename in unique_files:
    print('   -', filename)


✓ Модели и документы загружены
  Chunks: 350
  Files:  4
   - cbr_ai_ethics_code_2025.pdf
   - cbr_ai_finmarket.html
   - wiki_ai_in_wikimedia.txt
   - wiki_iskusstvennyy_intellekt.txt


In [6]:
pd.DataFrame(
    [
        {
            'chunk_id': doc.metadata.get('chunk_id'),
            'filename': doc.metadata.get('filename'),
            'page': doc.metadata.get('page'),
            'chars': len(doc.page_content),
        }
        for doc in all_docs[:10]
    ]
)


,chunk_id,filename,page,chars
0,cbr_ai_ethics_pdf:p1:c0:604b21eb9e1c,cbr_ai_ethics_code_2025.pdf,1,94
1,cbr_ai_ethics_pdf:p1:c1:9dbdfda66296,cbr_ai_ethics_code_2025.pdf,1,482
2,cbr_ai_ethics_pdf:p1:c2:6138da3e70b1,cbr_ai_ethics_code_2025.pdf,1,414
3,cbr_ai_ethics_pdf:p1:c3:84653a71a06d,cbr_ai_ethics_code_2025.pdf,1,498
4,cbr_ai_ethics_pdf:p1:c4:5c0c2685e152,cbr_ai_ethics_code_2025.pdf,1,257
5,cbr_ai_ethics_pdf:p2:c0:b9c7800dd6e5,cbr_ai_ethics_code_2025.pdf,2,94
6,cbr_ai_ethics_pdf:p2:c1:646c429eb47f,cbr_ai_ethics_code_2025.pdf,2,482
7,cbr_ai_ethics_pdf:p2:c2:5cd182348d8e,cbr_ai_ethics_code_2025.pdf,2,485
8,cbr_ai_ethics_pdf:p2:c3:efd515acac43,cbr_ai_ethics_code_2025.pdf,2,238
9,cbr_ai_ethics_pdf:p2:c4:c139a48f9ce8,cbr_ai_ethics_code_2025.pdf,2,483


## 1. Вспомогательные функции

Нам нужны четыре группы helper-функций:
- нормализация сущностей и парсинг JSON-ответов;
- flat RAG helper для сравнения;
- extraction helper для graph construction;
- GraphRAG retrieval helper поверх `Kuzu`.


In [7]:
def normalize_entity_name(name: str) -> str:
    cleaned = re.sub(r"\n+", " ", name.strip().lower())
    cleaned = re.sub(r"[^a-zA-Z0-9а-яА-ЯёЁ]+", "_", cleaned)
    cleaned = re.sub(r"_+", "_", cleaned).strip("_")
    return cleaned


def chunk_label(doc: Document) -> str:
    meta = doc.metadata
    return f"{meta.get('filename')}|p{meta.get('page')}|{meta.get('chunk_id')}"


def strip_json_fence(text: str) -> str:
    text = text.strip()
    if text.startswith('```json'):
        text = text[7:]
    elif text.startswith('```'):
        text = text[3:]
    if text.endswith('```'):
        text = text[:-3]
    return text.strip()


def parse_json_payload(text: str) -> dict[str, Any]:
    candidate = strip_json_fence(text)
    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", candidate, re.S)
        if not match:
            return {'entities': [], 'relations': []}
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            return {'entities': [], 'relations': []}


def show_docs(docs: list[Document], title: str | None = None, max_chars: int = 220) -> None:
    if title:
        print(title)
    for i, doc in enumerate(docs, 1):
        print(f'--- Result {i} ---')
        print(f"file={doc.metadata.get('filename')} | page={doc.metadata.get('page')} | chunk_id={doc.metadata.get('chunk_id')}")
        preview = doc.page_content.replace('\n', ' ')
        print(preview[:max_chars] + ('...' if len(preview) > max_chars else ''))
        print()


def docs_to_context(docs: list[Document], max_chars_per_doc: int = 700) -> str:
    blocks = []
    for doc in docs:
        blocks.append(
            f"[{chunk_label(doc)}]\n{doc.page_content[:max_chars_per_doc].replace(chr(10), ' ')}"
        )
    return '\n\n'.join(blocks)


def answer_from_docs(question: str, docs: list[Document]) -> str:
    messages = [
        SystemMessage(
            content=(
                'Ответь только по предоставленному контексту. '
                'Если данных недостаточно, скажи об этом явно. '
                'После ключевых фактов указывай citations вида [filename|page|chunk_id].'
            )
        ),
        HumanMessage(content=f"Вопрос:\n{question}\n\nКонтекст:\n{docs_to_context(docs)}"),
    ]
    response = llm.invoke(messages)
    return response.content if hasattr(response, 'content') else str(response)


def run_vector_search(query: str, k: int = 5) -> tuple[list[Document], list[float]]:
    results = vectorstore.similarity_search_with_score(query, k=k)
    return [doc for doc, _ in results], [score for _, score in results]


def run_bm25_search(query: str, k: int = 5) -> list[Document]:
    bm25_retriever.k = k
    return bm25_retriever.invoke(query)


def reciprocal_rank_fusion(rankings: list[list[Document]], k: int = 60) -> list[tuple[Document, float]]:
    best_doc_by_key: dict[str, Document] = {}
    scores: dict[str, float] = {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking, start=1):
            key = doc.metadata.get('chunk_id', chunk_label(doc))
            best_doc_by_key[key] = doc
            scores[key] = scores.get(key, 0.0) + 1.0 / (k + rank)
    fused = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    return [(best_doc_by_key[key], score) for key, score in fused]


def run_hybrid_search(query: str, k: int = 5) -> tuple[list[Document], list[float]]:
    vector_docs, _ = run_vector_search(query, k=k)
    bm25_docs = run_bm25_search(query, k=k)
    fused = reciprocal_rank_fusion([vector_docs, bm25_docs])[:k]
    return [doc for doc, _ in fused], [score for _, score in fused]


## 2. Извлечение сущностей и связей

Здесь мы просим модель для каждого чанка вернуть небольшой JSON:
- `entities`
- `relations`

Это не industrial-grade ontology. Нам нужна минимальная, но usable-after-course схема.

Чтобы не делать полный re-run каждый раз, extraction кэшируется в `graphrag_extractions.json`.

> Для занятия по умолчанию извлекаем граф не из всех 350 чанков, а из ограниченного подмножества (`GRAPH_EXTRACTION_MAX_CHUNKS`).
> Полный прогон делает один LLM-вызов на каждый чанк и может занять долгое время. Если нужен полный граф, выставьте `GRAPH_EXTRACTION_MAX_CHUNKS=0` в `.env` и перезапустите kernel.


In [8]:
EXTRACTION_SCHEMA_EXAMPLE = {
    'entities': [
        {'name': 'Банк России', 'type': 'Regulator'},
        {'name': 'искусственный интеллект', 'type': 'Technology'},
    ],
    'relations': [
        {
            'source': 'Банк России',
            'relation': 'регулирует применение',
            'target': 'искусственный интеллект',
            'evidence': 'Банк России придерживается риск-ориентированного подхода...'
        }
    ],
}


def extract_graph_payload(doc: Document) -> dict[str, Any]:
    prompt = (
        'Извлеки из текста сущности и связи между ними. '
        'Верни только JSON-объект без пояснений.\n\n'
        'Требования:\n'
        '1. Добавляй только сущности, которые явно есть в тексте.\n'
        '2. Связи должны быть короткими и явными: регулирует, применяет, использует, связан_с, снижает_риск, требует, включает и т.п.\n'
        '3. evidence должно быть короткой цитатоподобной выдержкой из текста, но без лишней длины.\n'
        '4. Не придумывай связей, если они не выражены явно.\n'
        f'5. Формат: {json.dumps(EXTRACTION_SCHEMA_EXAMPLE, ensure_ascii=False)}\n\n'
        f'Текст чанка:\n{doc.page_content[:2200]}'
    )
    response = llm.invoke(prompt)
    payload = parse_json_payload(response.content if hasattr(response, 'content') else str(response))

    entities = []
    for entity in payload.get('entities', []):
        name = str(entity.get('name', '')).strip()
        if not name:
            continue
        entities.append({
            'id': normalize_entity_name(name),
            'name': name,
            'type': str(entity.get('type', 'Unknown')).strip() or 'Unknown',
        })

    known_names = {entity['name'] for entity in entities}
    relations = []
    for rel in payload.get('relations', []):
        source = str(rel.get('source', '')).strip()
        target = str(rel.get('target', '')).strip()
        relation = str(rel.get('relation', '')).strip()
        if not source or not target or not relation:
            continue
        if source not in known_names:
            entities.append({'id': normalize_entity_name(source), 'name': source, 'type': 'Unknown'})
            known_names.add(source)
        if target not in known_names:
            entities.append({'id': normalize_entity_name(target), 'name': target, 'type': 'Unknown'})
            known_names.add(target)
        relations.append({
            'source_id': normalize_entity_name(source),
            'source_name': source,
            'relation': relation,
            'target_id': normalize_entity_name(target),
            'target_name': target,
            'evidence': str(rel.get('evidence', '')).strip(),
            'source_chunk_id': doc.metadata.get('chunk_id'),
        })

    return {
        'chunk_id': doc.metadata.get('chunk_id'),
        'filename': doc.metadata.get('filename'),
        'page': doc.metadata.get('page'),
        'entities': entities,
        'relations': relations,
    }


In [9]:
def select_graph_extraction_docs(docs: list[Document], max_chunks: int) -> list[Document]:
    """Возвращает подмножество чанков для учебного GraphRAG-прогона."""
    if max_chunks <= 0 or max_chunks >= len(docs):
        return docs

    # Берём чанки равномерно по файлам, чтобы graph demo не состоял только из первых страниц PDF.
    by_file: dict[str, list[Document]] = {}
    for doc in docs:
        by_file.setdefault(doc.metadata.get('filename', 'unknown'), []).append(doc)

    selected: list[Document] = []
    per_file = max(1, max_chunks // max(1, len(by_file)))
    for filename in sorted(by_file):
        selected.extend(by_file[filename][:per_file])

    # Если из-за округления набрали меньше лимита, добираем из оставшихся чанков в исходном порядке.
    selected_ids = {doc.metadata.get('chunk_id') for doc in selected}
    for doc in docs:
        if len(selected) >= max_chunks:
            break
        if doc.metadata.get('chunk_id') not in selected_ids:
            selected.append(doc)
            selected_ids.add(doc.metadata.get('chunk_id'))

    return selected[:max_chunks]


graph_docs = select_graph_extraction_docs(all_docs, GRAPH_EXTRACTION_MAX_CHUNKS)
write_notebook_status('graph_extraction', f'preparing {len(graph_docs)} chunks out of {len(all_docs)}')
print(f'Graph extraction docs: {len(graph_docs)} / {len(all_docs)}')
print('Files in graph extraction set:')
for filename in sorted({doc.metadata.get('filename', 'unknown') for doc in graph_docs}):
    count = sum(1 for doc in graph_docs if doc.metadata.get('filename') == filename)
    print(f'  - {filename}: {count} chunks')

if EXTRACTIONS_CACHE_PATH.exists():
    with open(EXTRACTIONS_CACHE_PATH, 'r', encoding='utf-8') as f:
        extracted_chunks = json.load(f)
    print(f'✓ Загружен extraction cache: {EXTRACTIONS_CACHE_PATH}')
else:
    extracted_chunks = []

extracted_by_chunk_id = {
    item.get('chunk_id'): item
    for item in extracted_chunks
    if item.get('chunk_id')
}

for doc in tqdm(graph_docs, desc='Extracting graph facts'):
    chunk_id = doc.metadata.get('chunk_id')
    if chunk_id in extracted_by_chunk_id:
        continue

    try:
        payload = extract_graph_payload(doc)
    except Exception as exc:
        payload = {
            'chunk_id': chunk_id,
            'filename': doc.metadata.get('filename'),
            'page': doc.metadata.get('page'),
            'entities': [],
            'relations': [],
            'error': f'{type(exc).__name__}: {exc}',
        }
        print(f'⚠ Extraction failed for {chunk_id}: {type(exc).__name__}: {exc}')

    extracted_by_chunk_id[chunk_id] = payload
    extracted_chunks = [extracted_by_chunk_id[doc.metadata.get('chunk_id')] for doc in graph_docs if doc.metadata.get('chunk_id') in extracted_by_chunk_id]

    # Checkpoint после каждого чанка: если kernel прервётся, уже готовые extraction не потеряются.
    with open(EXTRACTIONS_CACHE_PATH, 'w', encoding='utf-8') as f:
        json.dump(extracted_chunks, f, ensure_ascii=False, indent=2)

    # Текстовый progress нужен на случай, если tqdm widget в Jupyter/VS Code визуально зависнет.
    if len(extracted_chunks) % 5 == 0 or len(extracted_chunks) == len(graph_docs):
        progress_detail = f'processed {len(extracted_chunks)} / {len(graph_docs)} chunks'
        print(f'Processed {len(extracted_chunks)} / {len(graph_docs)} chunks')
        write_notebook_status('graph_extraction', progress_detail, processed=len(extracted_chunks), total=len(graph_docs))

errors = [item for item in extracted_chunks if item.get('error')]
print(f'Chunks with extraction payloads: {len(extracted_chunks)}')
print(f'Extraction errors: {len(errors)}')
write_notebook_status('graph_extraction', f'processed {len(extracted_chunks)} / {len(graph_docs)} chunks, errors={len(errors)}', status='done', processed=len(extracted_chunks), total=len(graph_docs), errors=len(errors))
if errors[:3]:
    print('Первые ошибки:')
    for item in errors[:3]:
        print(f"  - {item.get('chunk_id')}: {item.get('error')}")


▶ [graph_extraction] running: preparing 60 chunks out of 350
Graph extraction docs: 60 / 350
Files in graph extraction set:
  - cbr_ai_ethics_code_2025.pdf: 15 chunks
  - cbr_ai_finmarket.html: 15 chunks
  - wiki_ai_in_wikimedia.txt: 15 chunks
  - wiki_iskusstvennyy_intellekt.txt: 15 chunks
✓ Загружен extraction cache: /Users/Sergej/claude_code/agent_course/rag_course_repo/topic7_rag/data/graphrag_extractions.json


I0427 10:36:22.136356 77815639 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0427 10:36:22.141136 77827911 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(89, generation: 1)


Extracting graph facts:   0%|          | 0/60 [00:00<?, ?it/s]

Chunks with extraction payloads: 60
Extraction errors: 0
✓ [graph_extraction] done: processed 60 / 60 chunks, errors=0


In [10]:
entity_rows = []
relation_rows = []
for chunk in extracted_chunks:
    entity_rows.extend(chunk['entities'])
    relation_rows.extend(chunk['relations'])

entities_df = (
    pd.DataFrame(entity_rows)
    .drop_duplicates(subset=['id'])
    .sort_values(['type', 'name'])
    .reset_index(drop=True)
)
relations_df = (
    pd.DataFrame(relation_rows)
    .drop_duplicates(subset=['source_id', 'relation', 'target_id', 'source_chunk_id'])
    .reset_index(drop=True)
)

print(f'Entities:  {len(entities_df)}')
print(f'Relations: {len(relations_df)}')
entities_df.head(15)


Entities:  169
Relations: 188


,id,name,type
0,вандализм,вандализм,Activity
1,восприятие,восприятие,Activity
2,действия,действия,Activity
3,интеллектуальных_задач,интеллектуальных задач,Activity
4,обработка_естественного_языка,обработка естественного языка,Activity
5,планирование,планирование,Activity
6,поддержка_робототехники,поддержка робототехники,Activity
7,представление_знаний,представление знаний,Activity
8,рассуждение,рассуждение,Activity
9,алгоритм_предсказания_последовательностей,алгоритм предсказания последовательностей,Algorithm


In [11]:
relations_df.head(20)


,source_id,source_name,relation,target_id,target_name,evidence,source_chunk_id
0,банк_россии,Банк России,регулирует применение,искусственный_интеллект,искусственный интеллект,Банк России придерживается риск-ориентированно...,cbr_ai_ethics_pdf:p1:c0:604b21eb9e1c
1,банк_россии,Банк России,регулирует применение,искусственный_интеллект,искусственный интеллект,Банк России придерживается риск-ориентированно...,cbr_ai_ethics_pdf:p1:c1:9dbdfda66296
2,искусственный_интеллект,искусственный интеллект,применяется,кредитные_организации,кредитные организации,...к применению искусственного интеллекта кред...,cbr_ai_ethics_pdf:p1:c1:9dbdfda66296
3,искусственный_интеллект,искусственный интеллект,связан_с,финансовый_рынок,финансовый рынок,...оказывающими профессиональные услуги на фин...,cbr_ai_ethics_pdf:p1:c1:9dbdfda66296
4,искусственный_интеллект,искусственный интеллект,повышает доверие,клиенты,клиенты,повышение доверия физических и юридических лиц...,cbr_ai_ethics_pdf:p1:c1:9dbdfda66296
5,субъекты_национальной_платежной_системы,субъекты национальной платежной системы,применяют,искусственный_интеллект,искусственный интеллект,при оказании клиентам услуг,cbr_ai_ethics_pdf:p1:c2:6138da3e70b1
6,финансовый_рынок,финансовый рынок,включает,субъекты_национальной_платежной_системы,субъекты национальной платежной системы,субъектами национальной платежной системы (…) ...,cbr_ai_ethics_pdf:p1:c2:6138da3e70b1
7,финансовый_рынок,финансовый рынок,развивает,искусственный_интеллект,искусственный интеллект,содействие развитию искусственного интеллекта ...,cbr_ai_ethics_pdf:p1:c2:6138da3e70b1
8,финансовый_рынок,финансовый рынок,минимизирует риски,искусственный_интеллект,искусственный интеллект,"минимизация рисков, связанных с разработкой и ...",cbr_ai_ethics_pdf:p1:c2:6138da3e70b1
9,организации,организации,применяют,искусственный_интеллект,искусственный интеллект,для осуществления указанных целей при разработ...,cbr_ai_ethics_pdf:p1:c3:84653a71a06d


## 3. Построение property graph в `Kuzu`

Граф будет хранить:
- `Entity`
- `Chunk`
- `RELATES_TO`
- `MENTIONED_IN`

Это достаточно, чтобы:
- исследовать связи между сущностями;
- сохранять source grounding через `source_chunk_id`;
- потом собирать GraphRAG answers по retrieved graph facts.


In [12]:
write_notebook_status('kuzu_build', 'creating Kuzu database')

# Kuzu может создать database path как файл (+ .wal), а не как директорию.
# Поэтому cleanup должен корректно обрабатывать оба варианта.
if KUZU_DB_PATH.exists():
    if KUZU_DB_PATH.is_dir():
        shutil.rmtree(KUZU_DB_PATH)
    else:
        KUZU_DB_PATH.unlink()

kuzu_wal_path = Path(str(KUZU_DB_PATH) + '.wal')
if kuzu_wal_path.exists():
    if kuzu_wal_path.is_dir():
        shutil.rmtree(kuzu_wal_path)
    else:
        kuzu_wal_path.unlink()

kuzu_db = kuzu.Database(str(KUZU_DB_PATH))
conn = kuzu.Connection(kuzu_db)

conn.execute('CREATE NODE TABLE Entity(id STRING, name STRING, entity_type STRING, PRIMARY KEY(id));')
conn.execute('CREATE NODE TABLE Chunk(chunk_id STRING, filename STRING, page INT64, text STRING, PRIMARY KEY(chunk_id));')
conn.execute('CREATE REL TABLE RELATES_TO(FROM Entity TO Entity, relation STRING, evidence STRING, source_chunk_id STRING);')
conn.execute('CREATE REL TABLE MENTIONED_IN(FROM Entity TO Chunk, mention STRING);')

for row in entities_df.to_dict(orient='records'):
    conn.execute(
        'CREATE (:Entity {id: $id, name: $name, entity_type: $entity_type});',
        {
            'id': row['id'],
            'name': row['name'],
            'entity_type': row['type'],
        },
    )

for doc in all_docs:
    conn.execute(
        'CREATE (:Chunk {chunk_id: $chunk_id, filename: $filename, page: $page, text: $text});',
        {
            'chunk_id': doc.metadata.get('chunk_id'),
            'filename': doc.metadata.get('filename'),
            'page': int(doc.metadata.get('page', 0)),
            'text': doc.page_content[:4000],
        },
    )

for chunk in extracted_chunks:
    for entity in chunk['entities']:
        conn.execute(
            'MATCH (e:Entity {id: $entity_id}), (c:Chunk {chunk_id: $chunk_id}) '
            'CREATE (e)-[:MENTIONED_IN {mention: $mention}]->(c);',
            {
                'entity_id': entity['id'],
                'chunk_id': chunk['chunk_id'],
                'mention': entity['name'],
            },
        )

for rel in relations_df.to_dict(orient='records'):
    conn.execute(
        'MATCH (s:Entity {id: $source_id}), (t:Entity {id: $target_id}) '
        'CREATE (s)-[:RELATES_TO {relation: $relation, evidence: $evidence, source_chunk_id: $source_chunk_id}]->(t);',
        {
            'source_id': rel['source_id'],
            'target_id': rel['target_id'],
            'relation': rel['relation'],
            'evidence': rel['evidence'],
            'source_chunk_id': rel['source_chunk_id'],
        },
    )

print('✓ Graph построен в Kuzu')
write_notebook_status('kuzu_build', 'graph built in Kuzu', status='done')


▶ [kuzu_build] running: creating Kuzu database
✓ Graph построен в Kuzu
✓ [kuzu_build] done: graph built in Kuzu


In [13]:
node_summary = conn.execute(
    """
    MATCH (e:Entity)
    RETURN COUNT(e) AS entity_count;
    """
).get_as_df()

edge_summary = conn.execute(
    """
    MATCH ()-[r:RELATES_TO]->()
    RETURN COUNT(r) AS relation_count;
    """
).get_as_df()

print(node_summary)
print(edge_summary)


   entity_count
0           169
   relation_count
0             188


## 4. Базовые Cypher-запросы к графу

Прежде чем делать GraphRAG, полезно просто посмотреть на граф как на граф.


In [14]:
conn.execute(
    """
    MATCH (e:Entity)-[r:RELATES_TO]->()
    RETURN e.name AS entity, COUNT(r) AS out_degree
    ORDER BY out_degree DESC
    LIMIT 15;
    """
).get_as_df()


,entity,out_degree
0,искусственный интеллект,37
1,организации,29
2,Банк России,14
3,исследования искусственного интеллекта,7
4,машины,5
5,финансовый рынок,5
6,финансовые организации,4
7,Джон Маккарти,4
8,ИИ,4
9,организация,4


In [15]:
conn.execute(
    """
    MATCH (e:Entity {id: $entity_id})-[r:RELATES_TO]-(n:Entity)
    RETURN e.name AS seed, r.relation AS relation, n.name AS neighbor, r.source_chunk_id AS source_chunk_id
    ORDER BY neighbor;
    """,
    {'entity_id': normalize_entity_name('Банк России')},
).get_as_df()


,seed,relation,neighbor,source_chunk_id
0,Банк России,подготовил,Кодекс этики,cbr_ai_finmarket_web:p0:c7:35a9f3cb306d
1,Банк России,регулирует применение,искусственный интеллект,cbr_ai_finmarket_web:p0:c9:9c0cc577313c
2,Банк России,регулирует применение,искусственный интеллект,cbr_ai_finmarket_web:p0:c7:35a9f3cb306d
3,Банк России,обеспечивает условия для применения,искусственный интеллект,cbr_ai_finmarket_web:p0:c6:f397a5c4eb49
4,Банк России,формирует регуляторные инициативы,искусственный интеллект,cbr_ai_finmarket_web:p0:c6:f397a5c4eb49
5,Банк России,мониторит применение,искусственный интеллект,cbr_ai_finmarket_web:p0:c6:f397a5c4eb49
6,Банк России,регулирует применение,искусственный интеллект,cbr_ai_finmarket_web:p0:c3:839107f9958c
7,Банк России,регулирует применение,искусственный интеллект,cbr_ai_finmarket_web:p0:c0:07a1eda14964
8,Банк России,регулирует применение,искусственный интеллект,cbr_ai_ethics_pdf:p2:c0:b9c7800dd6e5
9,Банк России,регулирует применение,искусственный интеллект,cbr_ai_ethics_pdf:p1:c1:9dbdfda66296


In [16]:
conn.execute(
    """
    MATCH (a:Entity {id: $source_id})-[r:RELATES_TO]-(b:Entity {id: $target_id})
    RETURN a.name AS source, r.relation AS relation, b.name AS target, r.source_chunk_id AS source_chunk_id;
    """,
    {
        'source_id': normalize_entity_name('Банк России'),
        'target_id': normalize_entity_name('искусственный интеллект'),
    },
).get_as_df()


,source,relation,target,source_chunk_id
0,Банк России,регулирует применение,искусственный интеллект,cbr_ai_ethics_pdf:p1:c0:604b21eb9e1c
1,Банк России,регулирует применение,искусственный интеллект,cbr_ai_ethics_pdf:p1:c1:9dbdfda66296
2,Банк России,регулирует применение,искусственный интеллект,cbr_ai_ethics_pdf:p2:c0:b9c7800dd6e5
3,Банк России,регулирует применение,искусственный интеллект,cbr_ai_finmarket_web:p0:c0:07a1eda14964
4,Банк России,регулирует применение,искусственный интеллект,cbr_ai_finmarket_web:p0:c3:839107f9958c
5,Банк России,мониторит применение,искусственный интеллект,cbr_ai_finmarket_web:p0:c6:f397a5c4eb49
6,Банк России,формирует регуляторные инициативы,искусственный интеллект,cbr_ai_finmarket_web:p0:c6:f397a5c4eb49
7,Банк России,обеспечивает условия для применения,искусственный интеллект,cbr_ai_finmarket_web:p0:c6:f397a5c4eb49
8,Банк России,регулирует применение,искусственный интеллект,cbr_ai_finmarket_web:p0:c7:35a9f3cb306d
9,Банк России,регулирует применение,искусственный интеллект,cbr_ai_finmarket_web:p0:c9:9c0cc577313c


### Визуализация подграфа

Таблицы выше полезны для точных запросов, но для интуиции GraphRAG важно увидеть граф визуально.

Ниже строим небольшой интерактивный force-directed graph:
- узлы — сущности;
- рёбра — извлечённые отношения `RELATES_TO`;
- размер узла зависит от числа связей в выбранном подграфе;
- цвет узла зависит от `entity_type`.

Чтобы картинка оставалась читаемой, визуализируем не весь граф, а top relations. Параметр `MAX_GRAPH_EDGES` можно увеличить или уменьшить.

> Визуализация использует D3.js из CDN внутри HTML-ячейки и не добавляет Python-зависимостей в проект.


In [17]:
from IPython.display import HTML, display
import html
import json


def build_graph_viz_data(relations_df: pd.DataFrame, entities_df: pd.DataFrame, max_edges: int = 45) -> dict[str, list[dict[str, Any]]]:
    """Готовит компактный подграф для визуализации."""
    if relations_df.empty:
        return {"nodes": [], "links": []}

    entity_meta = entities_df.set_index('id').to_dict(orient='index') if not entities_df.empty else {}

    edge_df = (
        relations_df
        .groupby(['source_id', 'source_name', 'relation', 'target_id', 'target_name'], dropna=False)
        .agg(
            weight=('source_chunk_id', 'nunique'),
            example_chunk=('source_chunk_id', 'first'),
        )
        .reset_index()
        .sort_values(['weight', 'source_name', 'target_name'], ascending=[False, True, True])
        .head(max_edges)
    )

    degree: dict[str, int] = {}
    for row in edge_df.to_dict(orient='records'):
        degree[row['source_id']] = degree.get(row['source_id'], 0) + 1
        degree[row['target_id']] = degree.get(row['target_id'], 0) + 1

    nodes = []
    for node_id, node_degree in sorted(degree.items(), key=lambda item: (-item[1], item[0])):
        meta = entity_meta.get(node_id, {})
        nodes.append({
            'id': node_id,
            'name': meta.get('name') or node_id,
            'type': meta.get('type') or meta.get('entity_type') or 'Unknown',
            'degree': node_degree,
        })

    links = []
    for row in edge_df.to_dict(orient='records'):
        links.append({
            'source': row['source_id'],
            'target': row['target_id'],
            'relation': row['relation'],
            'weight': int(row['weight']),
            'example_chunk': row['example_chunk'],
        })

    return {'nodes': nodes, 'links': links}


def display_graph_viz(graph_data: dict[str, list[dict[str, Any]]], width: int = 980, height: int = 680) -> None:
    """Рисует интерактивный граф через D3 внутри iframe."""
    if not graph_data['nodes']:
        display(HTML('<p><b>Граф пуст:</b> нет relations для визуализации.</p>'))
        return

    payload = json.dumps(graph_data, ensure_ascii=False)
    html_doc = f"""
<!doctype html>
<html>
<head>
  <meta charset="utf-8" />
  <script src="https://cdn.jsdelivr.net/npm/d3@7"></script>
  <style>
    body {{ margin: 0; font-family: Inter, -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; background: #0f172a; }}
    .wrap {{ width: {width}px; height: {height}px; position: relative; overflow: hidden; background: radial-gradient(circle at top left, #1e3a8a 0, #0f172a 36%, #020617 100%); }}
    svg {{ width: 100%; height: 100%; }}
    .title {{ position: absolute; left: 18px; top: 14px; color: #e5e7eb; font-size: 15px; font-weight: 700; letter-spacing: .2px; }}
    .subtitle {{ position: absolute; left: 18px; top: 38px; color: #94a3b8; font-size: 12px; }}
    .legend {{ position: absolute; right: 18px; top: 14px; color: #cbd5e1; font-size: 12px; background: rgba(15, 23, 42, .68); border: 1px solid rgba(148, 163, 184, .22); border-radius: 14px; padding: 10px 12px; backdrop-filter: blur(8px); }}
    .legend-row {{ display: flex; align-items: center; gap: 7px; margin: 4px 0; white-space: nowrap; }}
    .dot {{ width: 9px; height: 9px; border-radius: 99px; display: inline-block; }}
    .link {{ stroke: rgba(148, 163, 184, .48); stroke-width: 1.4px; }}
    .node circle {{ stroke: rgba(255,255,255,.78); stroke-width: 1.2px; filter: drop-shadow(0 6px 12px rgba(0,0,0,.35)); cursor: grab; }}
    .node text {{ fill: #e5e7eb; font-size: 11px; paint-order: stroke; stroke: rgba(2, 6, 23, .92); stroke-width: 4px; stroke-linejoin: round; pointer-events: none; }}
    .edge-label {{ fill: #cbd5e1; font-size: 9px; opacity: .72; paint-order: stroke; stroke: rgba(2, 6, 23, .9); stroke-width: 3px; stroke-linejoin: round; pointer-events: none; }}
    .tooltip {{ position: absolute; max-width: 360px; padding: 10px 12px; border-radius: 12px; background: rgba(15, 23, 42, .94); color: #f8fafc; border: 1px solid rgba(148, 163, 184, .35); box-shadow: 0 18px 45px rgba(0,0,0,.38); pointer-events: none; opacity: 0; transition: opacity .12s ease; font-size: 12px; line-height: 1.35; }}
    .muted {{ color: #94a3b8; }}
  </style>
</head>
<body>
<div class="wrap">
  <div class="title">GraphRAG knowledge graph</div>
  <div class="subtitle">drag nodes · hover for details · top extracted relations</div>
  <div class="legend" id="legend"></div>
  <div class="tooltip" id="tooltip"></div>
  <svg viewBox="0 0 {width} {height}"></svg>
</div>
<script>
const graph = {payload};
const width = {width};
const height = {height};

const palette = ['#38bdf8', '#a78bfa', '#34d399', '#fbbf24', '#fb7185', '#60a5fa', '#f472b6', '#c084fc', '#22c55e', '#f97316'];
const types = Array.from(new Set(graph.nodes.map(d => d.type || 'Unknown'))).sort();
const color = new Map(types.map((type, i) => [type, palette[i % palette.length]]));

const legend = d3.select('#legend');
legend.append('div').style('font-weight', 700).style('margin-bottom', '6px').text('Entity types');
for (const type of types.slice(0, 10)) {{
  const row = legend.append('div').attr('class', 'legend-row');
  row.append('span').attr('class', 'dot').style('background', color.get(type));
  row.append('span').text(type);
}}

const svg = d3.select('svg');
const tooltip = d3.select('#tooltip');

svg.append('defs').append('marker')
  .attr('id', 'arrow')
  .attr('viewBox', '0 -5 10 10')
  .attr('refX', 20)
  .attr('refY', 0)
  .attr('markerWidth', 6)
  .attr('markerHeight', 6)
  .attr('orient', 'auto')
  .append('path')
  .attr('fill', 'rgba(148, 163, 184, .62)')
  .attr('d', 'M0,-5L10,0L0,5');

const link = svg.append('g')
  .selectAll('line')
  .data(graph.links)
  .join('line')
  .attr('class', 'link')
  .attr('marker-end', 'url(#arrow)')
  .attr('stroke-width', d => Math.min(4, 1.1 + d.weight * 0.45));

const edgeLabel = svg.append('g')
  .selectAll('text')
  .data(graph.links)
  .join('text')
  .attr('class', 'edge-label')
  .text(d => d.relation.length > 28 ? d.relation.slice(0, 26) + '…' : d.relation);

const node = svg.append('g')
  .selectAll('g')
  .data(graph.nodes)
  .join('g')
  .attr('class', 'node')
  .call(d3.drag()
    .on('start', dragstarted)
    .on('drag', dragged)
    .on('end', dragended));

node.append('circle')
  .attr('r', d => Math.min(20, 7 + d.degree * 1.4))
  .attr('fill', d => color.get(d.type || 'Unknown'));

node.append('text')
  .attr('x', d => Math.min(24, 12 + d.degree * 1.2))
  .attr('y', 4)
  .text(d => d.name.length > 34 ? d.name.slice(0, 32) + '…' : d.name);

node.on('mousemove', (event, d) => {{
    tooltip.style('opacity', 1)
      .style('left', Math.min(event.offsetX + 16, width - 380) + 'px')
      .style('top', (event.offsetY + 16) + 'px')
      .html(`<b>${{d.name}}</b><br><span class="muted">type:</span> ${{d.type}}<br><span class="muted">degree in view:</span> ${{d.degree}}`);
  }})
  .on('mouseleave', () => tooltip.style('opacity', 0));

link.on('mousemove', (event, d) => {{
    tooltip.style('opacity', 1)
      .style('left', Math.min(event.offsetX + 16, width - 380) + 'px')
      .style('top', (event.offsetY + 16) + 'px')
      .html(`<b>${{d.source.name || d.source}}</b> → <b>${{d.target.name || d.target}}</b><br><span class="muted">relation:</span> ${{d.relation}}<br><span class="muted">evidence chunks:</span> ${{d.weight}}<br><span class="muted">example:</span> ${{d.example_chunk}}`);
  }})
  .on('mouseleave', () => tooltip.style('opacity', 0));

const simulation = d3.forceSimulation(graph.nodes)
  .force('link', d3.forceLink(graph.links).id(d => d.id).distance(d => 90 + Math.min(80, d.relation.length * 1.4)).strength(0.42))
  .force('charge', d3.forceManyBody().strength(-460))
  .force('center', d3.forceCenter(width / 2, height / 2))
  .force('collide', d3.forceCollide().radius(d => Math.min(30, 16 + d.degree * 1.5)))
  .force('x', d3.forceX(width / 2).strength(0.035))
  .force('y', d3.forceY(height / 2).strength(0.035));

simulation.on('tick', () => {{
  link
    .attr('x1', d => d.source.x)
    .attr('y1', d => d.source.y)
    .attr('x2', d => d.target.x)
    .attr('y2', d => d.target.y);

  edgeLabel
    .attr('x', d => (d.source.x + d.target.x) / 2)
    .attr('y', d => (d.source.y + d.target.y) / 2);

  node.attr('transform', d => `translate(${{d.x}},${{d.y}})`);
}});

function dragstarted(event, d) {{
  if (!event.active) simulation.alphaTarget(0.3).restart();
  d.fx = d.x;
  d.fy = d.y;
}}
function dragged(event, d) {{
  d.fx = event.x;
  d.fy = event.y;
}}
function dragended(event, d) {{
  if (!event.active) simulation.alphaTarget(0);
  d.fx = null;
  d.fy = null;
}}
</script>
</body>
</html>
"""
    display(HTML(f'<iframe srcdoc="{html.escape(html_doc)}" width="{width}" height="{height}" style="border:0;border-radius:16px;box-shadow:0 18px 50px rgba(15,23,42,.18);"></iframe>'))


write_notebook_status('graph_visualization', 'building D3 subgraph')
MAX_GRAPH_EDGES = 45
graph_viz_data = build_graph_viz_data(relations_df, entities_df, max_edges=MAX_GRAPH_EDGES)
print(f"Visualizing {len(graph_viz_data['nodes'])} nodes and {len(graph_viz_data['links'])} relations")
display_graph_viz(graph_viz_data)
write_notebook_status('graph_visualization', f"rendered {len(graph_viz_data['nodes'])} nodes and {len(graph_viz_data['links'])} relations", status='done')


▶ [graph_visualization] running: building D3 subgraph
Visualizing 45 nodes and 45 relations


/Users/Sergej/claude_code/agent_course/rag_course_repo/.venv/lib/python3.12/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


✓ [graph_visualization] done: rendered 45 nodes and 45 relations


## 5. Graph retrieval helpers

Теперь превращаем граф в retrieval layer.

Pipeline будет очень прозрачным:
1. извлекаем из вопроса стартовые сущности;
2. находим вокруг них relations и supporting chunks;
3. собираем evidence bundle;
4. просим LLM ответить уже по graph evidence.


In [18]:
QUESTION_ENTITY_EXAMPLE = {
    'entities': ['Банк России', 'искусственный интеллект']
}


def extract_question_entities(question: str) -> list[str]:
    prompt = (
        'Выдели из вопроса сущности, которые стоит использовать как стартовые узлы для поиска в knowledge graph. '
        'Верни только JSON вида ' + json.dumps(QUESTION_ENTITY_EXAMPLE, ensure_ascii=False) + '.\n\n'
        f'Вопрос: {question}'
    )
    response = llm.invoke(prompt)
    payload = parse_json_payload(response.content if hasattr(response, 'content') else str(response))
    entities = [str(item).strip() for item in payload.get('entities', []) if str(item).strip()]
    return entities


def fetch_one_hop_facts(entity_name: str, limit: int = 20) -> list[dict[str, Any]]:
    entity_id = normalize_entity_name(entity_name)
    df = conn.execute(
        """
        MATCH (e:Entity {id: $entity_id})-[r:RELATES_TO]-(n:Entity)
        RETURN
            e.id AS seed_id,
            e.name AS seed_name,
            e.entity_type AS seed_type,
            r.relation AS relation,
            n.id AS neighbor_id,
            n.name AS neighbor_name,
            n.entity_type AS neighbor_type,
            r.evidence AS evidence,
            r.source_chunk_id AS source_chunk_id
        LIMIT $limit;
        """,
        {'entity_id': entity_id, 'limit': limit},
    ).get_as_df()
    return df.to_dict(orient='records')


def fetch_chunks_by_ids(chunk_ids: list[str]) -> list[Document]:
    docs_by_id = {doc.metadata.get('chunk_id'): doc for doc in all_docs}
    return [docs_by_id[chunk_id] for chunk_id in chunk_ids if chunk_id in docs_by_id]


def graph_retrieve(question: str, per_entity_limit: int = 12) -> dict[str, Any]:
    seed_entities = extract_question_entities(question)
    facts = []
    for entity in seed_entities:
        facts.extend(fetch_one_hop_facts(entity, limit=per_entity_limit))

    deduped = {}
    for fact in facts:
        key = (
            fact['seed_name'],
            fact['relation'],
            fact['neighbor_name'],
            fact['source_chunk_id'],
        )
        deduped[key] = fact

    facts = list(deduped.values())
    chunk_ids = list(dict.fromkeys([fact['source_chunk_id'] for fact in facts if fact.get('source_chunk_id')]))
    supporting_docs = fetch_chunks_by_ids(chunk_ids)

    return {
        'seed_entities': seed_entities,
        'facts': facts,
        'supporting_docs': supporting_docs,
    }


def facts_to_context(facts: list[dict[str, Any]]) -> str:
    lines = []
    for i, fact in enumerate(facts, 1):
        lines.append(
            f"{i}. {fact['seed_name']} --{fact['relation']}--> {fact['neighbor_name']} "
            f"| evidence={fact['evidence']} | chunk_id={fact['source_chunk_id']}"
        )
    return '\n'.join(lines)


def answer_from_graph(question: str, graph_bundle: dict[str, Any]) -> str:
    facts_context = facts_to_context(graph_bundle['facts'])
    messages = [
        SystemMessage(
            content=(
                'Ответь только по graph evidence. '
                'Если evidence недостаточно, скажи об этом явно. '
                'После ключевых фактов добавляй ссылки на source_chunk_id.'
            )
        ),
        HumanMessage(
            content=(
                f"Вопрос:\n{question}\n\n"
                f"Стартовые сущности:\n{graph_bundle['seed_entities']}\n\n"
                f"Graph facts:\n{facts_context}"
            )
        ),
    ]
    response = llm.invoke(messages)
    return response.content if hasattr(response, 'content') else str(response)


## 6. Первый GraphRAG demo

Сначала возьмем вопрос, который естественно ложится на relation retrieval.


In [19]:
graph_question_1 = 'Как связаны Банк России, искусственный интеллект и управление рисками?'
graph_bundle_1 = graph_retrieve(graph_question_1)

print('Стартовые сущности:', graph_bundle_1['seed_entities'])
print('\nGraph facts:\n')
for fact in graph_bundle_1['facts'][:12]:
    print(f"- {fact['seed_name']} --{fact['relation']}--> {fact['neighbor_name']} | {fact['source_chunk_id']}")
    if fact.get('evidence'):
        print(f"  evidence: {fact['evidence']}")
print('\nSupporting chunks:\n')
show_docs(graph_bundle_1['supporting_docs'][:5])


Стартовые сущности: ['Банк России', 'искусственный интеллект', 'управление рисками']

Graph facts:

- Банк России --подготовил--> Кодекс этики | cbr_ai_finmarket_web:p0:c7:35a9f3cb306d
  evidence: Для повышения доверия Банк России подготовил Кодекс этики...
- Банк России --провел опрос--> финансовый рынок | cbr_ai_finmarket_web:p0:c4:cb12b577a765
  evidence: По результатам опроса Банка России...
- Банк России --регулирует--> финансовый рынок | cbr_ai_finmarket_web:p0:c1:e2b46f3368ca
  evidence: Обратиться в Банк России
- Банк России --выразили готовность работать совместно с--> участники финансового рынка | cbr_ai_finmarket_web:p0:c8:fbb2d4859fff
  evidence: Участники финансового рынка выразили готовность совместно с Банком России...
- Банк России --регулирует применение--> искусственный интеллект | cbr_ai_finmarket_web:p0:c9:9c0cc577313c
  evidence: Банк России придерживается риск-ориентированного подхода...
- Банк России --регулирует применение--> искусственный интеллект | cbr_ai_fin

In [20]:
graph_answer_1 = answer_from_graph(graph_question_1, graph_bundle_1)
print(graph_answer_1)


Связь между Банком России, искусственным интеллектом (ИИ) и управлением рисками можно проследить через несколько граф-факторов.

1. **Регулирование ИИ**: Банк России регулирует применение искусственного интеллекта на финансовом рынке, придерживаясь риск-ориентированного подхода. Это означает, что он контролирует использование технологий ИИ участниками рынка с целью минимизации рисков. [chunk_id=cbr_ai_finmarket_web:p0:c9:9c0cc577313c]
   
2. **Мониторинг использования ИИ**: Банк России осуществляет мониторинг применения искусственного интеллекта, чтобы своевременно выявлять потенциальные риски и принимать меры по их снижению. [chunk_id=cbr_ai_finmarket_web:p0:c6:f397a5c4eb49]

Таким образом, связь заключается в том, что Банк России активно участвует в регулировании и мониторинге использования искусственного интеллекта в финансовой отрасли, ориентируясь на снижение возможных рисков. Однако конкретные примеры или детали того, как именно ИИ применяется для управления рисками, не представл

## 7. GraphRAG vs flat RAG

Теперь сравним graph retrieval с hybrid flat RAG из предыдущего ноутбука.

Нас интересуют relation-heavy вопросы, где полезно явно видеть связи между сущностями.


In [21]:
comparison_questions = [
    'Как связаны Банк России, искусственный интеллект и управление рисками?',
    'Как связаны искусственный интеллект, персональные данные и качество наборов данных?',
]
comparison_questions


['Как связаны Банк России, искусственный интеллект и управление рисками?',
 'Как связаны искусственный интеллект, персональные данные и качество наборов данных?']

In [22]:
comparison_rows = []
for question in comparison_questions:
    flat_docs, _ = run_hybrid_search(question, k=5)
    flat_answer = answer_from_docs(question, flat_docs)

    graph_bundle = graph_retrieve(question)
    graph_answer = answer_from_graph(question, graph_bundle)

    comparison_rows.append({
        'question': question,
        'flat_docs': len(flat_docs),
        'graph_seed_entities': ', '.join(graph_bundle['seed_entities']),
        'graph_fact_count': len(graph_bundle['facts']),
        'flat_answer': flat_answer[:500],
        'graph_answer': graph_answer[:500],
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df[['question', 'flat_docs', 'graph_seed_entities', 'graph_fact_count']]


,question,flat_docs,graph_seed_entities,graph_fact_count
0,"Как связаны Банк России, искусственный интелле...",5,"Банк России, искусственный интеллект, управлен...",24
1,"Как связаны искусственный интеллект, персональ...",5,"искусственный интеллект, персональные данные, ...",12


In [23]:
question = comparison_questions[1]
print('QUESTION:\n', question)

flat_docs, _ = run_hybrid_search(question, k=5)
print('\n=== Flat RAG retrieved docs ===\n')
show_docs(flat_docs)
print('\n=== Flat RAG answer ===\n')
print(answer_from_docs(question, flat_docs))

graph_bundle = graph_retrieve(question)
print('\n=== GraphRAG facts ===\n')
for fact in graph_bundle['facts'][:15]:
    print(f"- {fact['seed_name']} --{fact['relation']}--> {fact['neighbor_name']} | {fact['source_chunk_id']}")
    if fact.get('evidence'):
        print(f"  evidence: {fact['evidence']}")
print('\n=== GraphRAG answer ===\n')
print(answer_from_graph(question, graph_bundle))


QUESTION:
 Как связаны искусственный интеллект, персональные данные и качество наборов данных?

=== Flat RAG retrieved docs ===

--- Result 1 ---
file=cbr_ai_ethics_code_2025.pdf | page=6 | chunk_id=cbr_ai_ethics_pdf:p6:c4:a2129737ba4e
предмет	наличия	рисков	искусственного	 интеллекта

--- Result 2 ---
file=wiki_iskusstvennyy_intellekt.txt | page=0 | chunk_id=wiki_artificial_intelligence:p0:c144:d4bb22ff211f
. Он использует как исторические данные, так и данные в реальном времени, чтобы понять, что сработало хорошо в прошлом, а также то, что в настоящее время имеет тенденцию в Интернете. Другая компания, называемая Yseop, ис...

--- Result 3 ---
file=cbr_ai_ethics_code_2025.pdf | page=7 | chunk_id=cbr_ai_ethics_pdf:p7:c1:5daf7ab2b347
7)	 	использование	наборов	данных,	полученных	от третьих	лиц,	или	 наборов	данных,	находящихся	в открытом	доступе	в информационнотелекоммуникационной	сети	«Интернет»; 8)	 	наличие	риск-событий,	связанных	с применением	ис...

--- Result 4 ---
file=wiki_isku

## 8. Что здесь важно методически

Этот ноутбук не пытается доказать, что GraphRAG всегда лучше flat RAG.

Идея ровно в другом:
- flat RAG удобен, когда ответ лежит в похожих текстовых чанках;
- GraphRAG удобен, когда нужно явно пройти по сущностям и связям между ними.

То есть это дополнительная технология, а не универсальная замена vector retrieval.


## 8. Phoenix status

Если tracing включен, extraction и generation из этого ноутбука пишутся в отдельный Phoenix project.


In [24]:
print('Phoenix enabled:', PHOENIX_ENABLED)
if PHOENIX_ENABLED:
    print('Phoenix project:', PHOENIX_PROJECT_NAME)
    print('Phoenix working dir:', PHOENIX_WORKING_DIR)
    print('Collector endpoint:', PHOENIX_COLLECTOR_ENDPOINT or f'local -> http://127.0.0.1:{PHOENIX_PORT}/v1/traces')
    if PHOENIX_SESSION_URL:
        print('Phoenix UI:', PHOENIX_SESSION_URL)
else:
    print('Phoenix не активирован в текущем окружении.')


Phoenix enabled: True
Phoenix project: topic7-rag-03
Phoenix working dir: ./.phoenix
Collector endpoint: local -> http://127.0.0.1:6006/v1/traces
Phoenix UI: http://localhost:6006/


## 9. Landscape: другие стеки и фреймворки

Если захочется развивать эту тему дальше, вот полезные направления:

### Graph databases
- `Kuzu` — embedded property-graph DB с Cypher: https://docs.kuzudb.com/
- `Neo4j` — production-style graph DB: https://neo4j.com/docs/
- `FalkorDBLite` — lightweight graph DB option: https://github.com/FalkorDB/falkordblite

### GraphRAG frameworks
- `LightRAG` — готовый GraphRAG framework: https://github.com/HKUDS/LightRAG
- `Microsoft GraphRAG` — advanced indexing/query pipeline: https://github.com/microsoft/graphrag

### RDF / semantic graph stack
- `RDFLib` — triples + SPARQL approach: https://rdflib.readthedocs.io/

В этом ноутбуке мы берем `Kuzu`, потому что он дает достаточно практичный, но еще прозрачный graph layer.


## Exercises

1. Добавь в graph schema дополнительные типы сущностей, например `Event`, `Model`, `Company`, `Person`.
2. Сделай двухшаговый graph retrieval: сначала seed entities, потом расширение через соседей второго порядка.
3. Сравни, какие вопросы flat RAG решает лучше, а какие — graph retrieval.
4. Добавь фильтрацию relations по confidence или relation whitelist.
5. Попробуй заменить `Kuzu` на `Neo4j` или `LightRAG` и сравни UX.


In [ ]:
# TODO / answer scaffold
NEXT_STEPS = {
    'better_relation_schema': False,
    'two_hop_retrieval': False,
    'relation_whitelist': False,
    'neo4j_or_lightrag_comparison': False,
}

NEXT_STEPS


## Итоги

В этом ноутбуке мы построили **GraphRAG на Kuzu** поверх всего текущего корпуса документов.

Ключевая мысль:
- `02` показывал retrieval по чанкам;
- `03` показывает retrieval по сущностям и связям.

Это не отменяет vector / hybrid RAG.
Но теперь у нас есть еще один практический путь, который можно использовать там, где знания естественно описываются через граф.


## Практическое упражнение: добрать граф по оставшимся чанкам

В базовом demo graph extraction может быть построен только на подмножестве корпуса: это ускоряет занятие, но оставляет часть чанков вне entity / relation extraction.

Задача студентов — **самостоятельно закрыть этот gap** и проверить, меняется ли GraphRAG-ответ после расширения графа.

Что нужно сделать:

1. Посчитать coverage: сколько чанков уже есть в `graphrag_extractions.json`, сколько осталось.
2. Выбрать остаток чанков и прогнать их через `extract_graph_payload()`.
3. Аккуратно обновить extraction cache без потери уже готовых результатов.
4. Пересобрать `entities_df`, `relations_df` и Kuzu-граф.
5. Повторить тот же вопрос и проверить groundedness ответа по graph facts.

> Для аудитории можно поставить небольшой лимит, например `REMAINING_EXTRACTION_LIMIT = 10`. Для полного добора используйте `0`.


In [ ]:
# Exercise 1. Coverage report
#
# Цель: получить таблицу по файлам:
# - chunks_total
# - chunks_extracted
# - chunks_remaining
# - coverage_pct

# TODO 1: соберите множество chunk_id, для которых уже есть extraction payload.
# Подсказка: используйте extracted_chunks и item.get('chunk_id').
extracted_ids = ...

# TODO 2: найдите документы из all_docs, которых еще нет в extracted_ids.
remaining_docs = ...

# TODO 3: соберите coverage_rows по каждому filename.
coverage_rows = []
for filename in sorted({doc.metadata.get('filename', 'unknown') for doc in all_docs}):
    file_docs = [doc for doc in all_docs if doc.metadata.get('filename', 'unknown') == filename]

    # TODO: посчитайте done для этого файла.
    done = ...

    coverage_rows.append({
        'filename': filename,
        'chunks_total': len(file_docs),
        'chunks_extracted': done,
        'chunks_remaining': len(file_docs) - done,
        'coverage_pct': round(100 * done / max(1, len(file_docs)), 1),
    })

coverage_df = pd.DataFrame(coverage_rows).sort_values(['chunks_remaining', 'filename'], ascending=[False, True])
print(f"Extraction coverage: {len(all_docs) - len(remaining_docs)} / {len(all_docs)} chunks")
print(f"Remaining chunks:    {len(remaining_docs)}")
display(coverage_df)

# Self-check: после заполнения TODO это должно выполняться без ошибки.
assert isinstance(remaining_docs, list)
assert {'chunks_total', 'chunks_extracted', 'chunks_remaining', 'coverage_pct'} <= set(coverage_df.columns)


In [ ]:
# Exercise 2. Прогон remaining chunks через entity / relation extraction
#
# Цель: добрать extraction payloads для оставшихся чанков и сохранить cache.
# Важно: не перезаписывайте уже готовые payloads пустым списком.

REMAINING_EXTRACTION_LIMIT = 10  # поменяйте на 0 для полного добора
REMAINING_CHECKPOINT_EVERY = 5
RUN_REMAINING_EXTRACTION = True

# TODO 1: загрузите свежий cache с диска, если EXTRACTIONS_CACHE_PATH существует.
# Иначе используйте текущий extracted_chunks.
cached_extractions = ...

# TODO 2: постройте словарь chunk_id -> payload.
extracted_by_chunk_id = ...

# TODO 3: заново посчитайте remaining_docs на основе extracted_by_chunk_id.
remaining_docs = ...

# TODO 4: выберите docs_to_extract с учетом REMAINING_EXTRACTION_LIMIT.
docs_to_extract = ...

print(f"Already extracted: {len(all_docs) - len(remaining_docs)} / {len(all_docs)}")
print(f"Will extract now:  {len(docs_to_extract)} chunks")

if RUN_REMAINING_EXTRACTION:
    for idx, doc in enumerate(tqdm(docs_to_extract, desc='Extracting remaining graph facts'), start=1):
        chunk_id = doc.metadata.get('chunk_id')

        # TODO 5: пропустите chunk_id, если он уже есть в extracted_by_chunk_id.

        # TODO 6: вызовите extract_graph_payload(doc).
        # Если вызов упал, сохраните payload с keys:
        # chunk_id, filename, page, entities=[], relations=[], error='...'
        payload = ...

        # TODO 7: положите payload в extracted_by_chunk_id.

        # TODO 8: каждые REMAINING_CHECKPOINT_EVERY чанков сохраняйте cache на диск.
        # Подсказка: сохраняйте payloads в порядке all_docs, чтобы diff был стабильнее.

# TODO 9: обновите переменную extracted_chunks из extracted_by_chunk_id.
extracted_chunks = ...

print('Done. Re-run Exercise 1 to check updated coverage.')


In [ ]:
# Exercise 3. Пересборка entities_df и relations_df из обновленного cache
#
# Цель: после добора чанков получить новые таблицы сущностей и связей.

entity_rows = []
relation_rows = []

# TODO 1: пройти по extracted_chunks и заполнить entity_rows / relation_rows.
# Подсказка: chunk.get('entities', []) и chunk.get('relations', []).

# TODO 2: собрать entities_df:
# - columns: id, name, type
# - dropna по id
# - drop_duplicates по id
# - sort_values по type/name
entities_df = ...

# TODO 3: собрать relations_df:
# - columns: source_id, source_name, relation, target_id, target_name, evidence, source_chunk_id
# - dropna по source_id/relation/target_id/source_chunk_id
# - drop_duplicates по source_id/relation/target_id/source_chunk_id
relations_df = ...

print(f'Entities after remaining extraction:  {len(entities_df)}')
print(f'Relations after remaining extraction: {len(relations_df)}')
display(entities_df.head(15))
display(relations_df.head(15))

assert not entities_df.empty, 'entities_df пустой — проверьте сбор entity_rows'
assert {'source_id', 'relation', 'target_id', 'source_chunk_id'} <= set(relations_df.columns)


In [ ]:
# Exercise 4. Пересборка Kuzu-графа
#
# Цель: создать новый Kuzu graph из обновленных entities_df / relations_df.
# Можно опираться на код из раздела "Построение property graph", но не копировать blindly:
# проверьте cleanup старого KUZU_DB_PATH и .wal, затем создайте таблицы и relations.

# TODO 1: корректно удалить старый KUZU_DB_PATH и KUZU_DB_PATH + '.wal', если они существуют.

# TODO 2: создать kuzu.Database и kuzu.Connection.
conn = ...

# TODO 3: создать node tables Entity / Chunk и relation tables RELATES_TO / MENTIONED_IN.

# TODO 4: загрузить entities_df в Entity.

# TODO 5: загрузить all_docs в Chunk.

# TODO 6: загрузить mentions из extracted_chunks в MENTIONED_IN.

# TODO 7: загрузить relations_df в RELATES_TO.

print('✓ Kuzu-граф пересобран по расширенному extraction cache')

# Self-check: в графе должны появиться узлы Entity и связи RELATES_TO.
entity_count = conn.execute('MATCH (e:Entity) RETURN count(e) AS n;').get_as_df()['n'].iloc[0]
relation_count = conn.execute('MATCH (:Entity)-[r:RELATES_TO]->(:Entity) RETURN count(r) AS n;').get_as_df()['n'].iloc[0]
print({'entities_in_graph': int(entity_count), 'relations_in_graph': int(relation_count)})
assert entity_count == len(entities_df)


In [ ]:
# Exercise 5. Повторная проверка ответа после расширения графа
#
# Цель: сравнить graph facts и ответ до/после добора remaining chunks.

exercise_question = 'Как связаны искусственный интеллект, персональные данные и качество наборов данных?'

# TODO 1: запустите flat retrieval для baseline.
flat_docs, _ = ...

# TODO 2: запустите graph_retrieve с увеличенным per_entity_limit.
full_graph_bundle = ...

# TODO 3: получите ответ через answer_from_graph.
full_graph_answer = ...

print('QUESTION:\n', exercise_question)
print('\n=== Flat RAG retrieved docs ===\n')
show_docs(flat_docs)

print('\n=== GraphRAG facts after remaining extraction ===\n')
# TODO 4: выведите первые 20-25 graph facts так же, как в основном demo.

print('\n=== GraphRAG answer after remaining extraction ===\n')
print(full_graph_answer)

# Student note:
# В markdown ниже ответа запишите, какие новые source_chunk_id появились
# и изменили ли они финальный ответ.


In [ ]:
# Exercise 6. Groundedness check
#
# Цель: проверить, поддержан ли финальный ответ найденными graph facts.
# Не нужно доказывать, что ответ идеален; нужно явно увидеть missing evidence / unsupported claims.

def verify_answer_against_graph(question: str, answer: str, graph_bundle: dict[str, Any]) -> str:
    """TODO: вернуть JSON-verdict по groundedness ответа."""
    # TODO 1: составьте SystemMessage для verifier-а.
    # Требуемый JSON shape:
    # {
    #   "grounded": true/false,
    #   "missing_evidence": [...],
    #   "unsupported_claims": [...],
    #   "suggested_fix": "..."
    # }

    # TODO 2: в HumanMessage передайте question, answer и facts_to_context(graph_bundle['facts']).

    # TODO 3: вызовите llm.invoke(...) и верните content.
    raise NotImplementedError


verification_report = verify_answer_against_graph(exercise_question, full_graph_answer, full_graph_bundle)
print('=== Groundedness check ===')
print(verification_report)


### Что студент должен сдать по этому упражнению

1. `coverage_df` до и после добора remaining chunks.
2. Количество новых `entities` и `relations` после пересборки таблиц.
3. 5–10 наиболее полезных graph facts для вопроса про ИИ, персональные данные и качество данных.
4. Финальный ответ GraphRAG после расширения графа.
5. Короткий groundedness verdict: какие утверждения подтверждены graph evidence, а где evidence всё еще не хватает.

Главная мысль упражнения: качество GraphRAG зависит не только от generation prompt, но и от полноты entity / relation extraction на уровне чанков.
